# Applied AI in Chemical and Process Engineering
## Convolutional Neural Networks (CNN) for Process Condition Monitoring
### Thermal Infrared Thermography for Heat Exchanger Fouling Detection

---

### Process Monitoring Context
In chemical plants and refineries, **heat exchanger fouling and furnace tube coking** restrict heat transfer, reduce thermal efficiency, and cause localized overheating that can lead to tube failure.

* **Computer Vision Approach:** Infrared (IR) thermography cameras capture 2D surface temperature maps of equipment.
* **Why CNNs?** Standard neural networks (MLPs) flatten 2D images into 1D vectors, losing spatial proximity and conduction gradients. CNNs use 2D sliding kernels to preserve spatial context and detect localized hotspots across tube surfaces.

```
[2D Thermal IR (64x64)] -> [Conv2D + MaxPool] -> [Conv2D + MaxPool] -> [Linear Head] -> Clean (0) vs. Fouled (1)
```

In [ ]:
# Environment setup and dependencies
import glob
import os
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Aesthetic plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 11

print(f"Setup complete. PyTorch Version: {torch.__version__}")


## Part 0: Images as Numerical Matrices

To a computer, an image is a numerical matrix (or 3D tensor):
* **Grayscale Image:** A 2D matrix $\mathbf{I} \in \mathbb{R}^{H \times W}$, where each entry is an intensity value from $0$ (black) to $255$ (white).
* **Color (RGB) Image:** Three stacked 2D matrices $\mathbf{I} \in \mathbb{R}^{H \times W \times 3}$ (Red, Green, Blue channels).
* **Thermal IR Image:** A 2D matrix of calibrated temperatures $T(i, j) \in [45^\circ\text{C}, 110^\circ\text{C}]$.

Because images are numerical matrices, image processing and computer vision are built on mathematical operations on data tables.

In [ ]:
# 1. Load Abraham Lincoln portrait and inspect its numerical matrix
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Cache image in data directory
data_dir = "data" if os.path.exists("data") else "../data"
os.makedirs(data_dir, exist_ok=True)
img_path = os.path.join(data_dir, "lincoln.jpg")
img_url = "https://upload.wikimedia.org/wikipedia/commons/a/ab/Abraham_Lincoln_O-77_matte_collodion_print.jpg"

if not os.path.exists(img_path):
    print("Downloading historical Abraham Lincoln portrait (O-77, 1863)...")
    req = urllib.request.Request(img_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(img_path, "wb") as f:
        f.write(resp.read())
    print("Download complete.")

# Convert to grayscale 2D intensity matrix
img_pil = Image.open(img_path).convert('L')
w_disp = 300
h_disp = int(w_disp * (img_pil.height / img_pil.width))
img_small = img_pil.resize((w_disp, h_disp), Image.Resampling.LANCZOS)
I_lincoln = np.array(img_small, dtype=np.float32)

print(f"Matrix shape: {I_lincoln.shape[0]}x{I_lincoln.shape[1]} pixels ({I_lincoln.size:,} values)")
print(f"Intensity range: min={I_lincoln.min():.0f}, max={I_lincoln.max():.0f}")

# Extract a 10x10 patch at Lincoln's eye
y1, y2, x1, x2 = 136, 146, 146, 156
patch = I_lincoln[y1:y2, x1:x2].astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

# (1) Full Image
axes[0].imshow(I_lincoln, cmap='gray')
rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='#e74c3c', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title("1. Image (Lincoln)", fontsize=11, fontweight='bold')
axes[0].axis('off')

# (2) Zoomed 10x10 Patch
axes[1].imshow(patch, cmap='gray', interpolation='nearest')
axes[1].set_title(f"2. Eye Region ({y2-y1}x{x2-x1} Patch)", fontsize=11, fontweight='bold')
axes[1].set_xticks(range(x2 - x1))
axes[1].set_yticks(range(y2 - y1))
axes[1].grid(color='red', linestyle='-', linewidth=0.5, alpha=0.5)

# (3) Numerical Table with Pixel Values
im = axes[2].imshow(patch, cmap='Blues_r', interpolation='nearest')
for i in range(patch.shape[0]):
    for j in range(patch.shape[1]):
        val = patch[i, j]
        text_color = 'white' if val < 90 else 'black'
        axes[2].text(j, i, f"{val}", ha='center', va='center', color=text_color, fontsize=8, fontweight='bold')
axes[2].set_title("3. Numerical Table (Pixel Values)", fontsize=11, fontweight='bold')
axes[2].set_xticks(range(x2 - x1))
axes[2].set_yticks(range(y2 - y1))

plt.suptitle("Image Representation: Pixel Matrix and Numerical Table", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Elementary Mathematical Operations on the Image Matrix
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))

# Original
axes[0, 0].imshow(I_lincoln, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title(r"1. Original Matrix: $\mathbf{I}$", fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

# Addition (Brightness)
I_bright = np.clip(I_lincoln + 60.0, 0, 255)
axes[0, 1].imshow(I_bright, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(r"2. Addition: $\mathbf{I} + 60$ (Brightness)", fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

# Inversion (Negative)
I_negative = 255.0 - I_lincoln
axes[0, 2].imshow(I_negative, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title(r"3. Inversion: $255 - \mathbf{I}$ (Negative)", fontsize=11, fontweight='bold')
axes[0, 2].axis('off')

# Contrast Scaling
I_contrast = np.clip((I_lincoln - 128.0) * 1.8 + 128.0, 0, 255)
axes[1, 0].imshow(I_contrast, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title(r"4. Scaling: $\alpha(\mathbf{I} - \mu) + \mu$ (Contrast)", fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

# Thresholding (Binarization)
I_thresh = (I_lincoln > 120.0).astype(np.float32) * 255.0
axes[1, 1].imshow(I_thresh, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title(r"5. Thresholding: $(\mathbf{I} > 120) \times 255$ (Binary Mask)", fontsize=11, fontweight='bold')
axes[1, 1].axis('off')

# Reflection (Flip)
I_flip = np.fliplr(I_lincoln)
axes[1, 2].imshow(I_flip, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title(r"6. Reflection: $\mathbf{I}[:, ::-1]$ (Horizontal Flip)", fontsize=11, fontweight='bold')
axes[1, 2].axis('off')

plt.suptitle("Mathematical Operations on the Image Matrix", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### Spatial Convolutions: The Foundation of CNNs

Single-pixel operations modify brightness or contrast, but detecting structure (edges, boundaries, hotspots) requires comparing a pixel with its neighbors via **spatial derivatives**:

$$\frac{\partial \mathbf{I}}{\partial x} \approx \mathbf{I}(i, j+1) - \mathbf{I}(i, j-1), \quad \frac{\partial \mathbf{I}}{\partial y} \approx \mathbf{I}(i+1, j) - \mathbf{I}(i-1, j)$$

A **2D convolution** computes these differences by sliding a small weight matrix ($3 \times 3$ kernel) across the image:
* **Classical Vision:** Kernels (e.g., Sobel filters) were hand-designed by engineers.
* **Convolutional Neural Networks (CNNs):** The kernel weights are **learned automatically** from data through backpropagation to detect relevant features (e.g., thermal hotspots on tubes).

In [ ]:
# 3. 2D Spatial Convolutions: Applying sliding kernels with PyTorch
import torch
import torch.nn.functional as F

t_lincoln = torch.tensor(I_lincoln, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Sobel-X: Computes dI/dx (detects vertical boundaries)
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).unsqueeze(0).unsqueeze(0)

# Sobel-Y: Computes dI/dy (detects horizontal boundaries)
sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).unsqueeze(0).unsqueeze(0)

# 2D Convolution (same operation as in CNN layers)
grad_x = F.conv2d(t_lincoln, sobel_x, padding=1).squeeze().numpy()
grad_y = F.conv2d(t_lincoln, sobel_y, padding=1).squeeze().numpy()
gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

axes[0].imshow(I_lincoln, cmap='gray')
axes[0].set_title(r"(a) Original $\mathbf{I}$", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(np.abs(grad_x), cmap='magma')
axes[1].set_title(r"(b) Vertical Edges $\mathbf{G}_x$ (Sobel-X)", fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(np.abs(grad_y), cmap='magma')
axes[2].set_title(r"(c) Horizontal Edges $\mathbf{G}_y$ (Sobel-Y)", fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(gradient_magnitude, cmap='inferno')
axes[3].set_title(r"(d) Gradient Magnitude $\sqrt{\mathbf{G}_x^2 + \mathbf{G}_y^2}$", fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle("2D Spatial Convolution: Extracting Feature Maps via Sliding Kernels", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("2D convolution applied: 3x3 kernels extract spatial derivative feature maps.")
print("In CNNs, these kernel weights are learned automatically during training.")

## Part 1: Loading Plant Thermal Inspection Images

The dataset in `data/heat-ex/` contains 600 thermal inspection scans (64x64 resolution):
* `clean/`: 300 scans with normal axial temperature gradients ($45^\circ\text{C} \to 80^\circ\text{C}$).
* `fouled/`: 300 scans showing localized overheating anomalies ($+20^\circ\text{C} - 28^\circ\text{C}$ hotspots) from internal coking or scale.

Images are normalized to $[0, 1]$ and split into 80% training (480 scans) and 20% test (120 scans).

In [ ]:
# Optional: Automated Download from GitHub (Useful for Google Colab or new machines)
import os
import urllib.request
import zipfile

data_dir = 'data/heat-ex'
if not os.path.exists(data_dir):
    print('Downloading heat exchanger thermal inspection scans from GitHub...')
    # Direct raw download URL:
    url = 'https://raw.githubusercontent.com/dissabnd/Applied-AI-CPE-UG-2026/main/data/heat-ex.zip'
    zip_path = 'heat-ex.zip'
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data/')
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print('Download and extraction complete: Dataset ready in data/heat-ex!')
else:
    print('Dataset already found locally in data/heat-ex.')


In [ ]:
# Load thermal inspection images from data/heat-ex directory
data_dir = "data/heat-ex" if os.path.exists("data/heat-ex") else "../data/heat-ex"

clean_paths = sorted(glob.glob(os.path.join(data_dir, "clean", "*.png")))
fouled_paths = sorted(glob.glob(os.path.join(data_dir, "fouled", "*.png")))

print(f"Found {len(clean_paths)} Clean scans and {len(fouled_paths)} Fouled scans.")

images = []
labels = []

# Load Clean scans (Label = 0)
for p in clean_paths:
    img = Image.open(p).convert('L')
    arr = np.array(img, dtype=np.float32) / 255.0  # Normalize to [0, 1]
    images.append(arr)
    labels.append(0)

# Load Fouled scans (Label = 1)
for p in fouled_paths:
    img = Image.open(p).convert('L')
    arr = np.array(img, dtype=np.float32) / 255.0
    images.append(arr)
    labels.append(1)

X = np.expand_dims(np.array(images, dtype=np.float32), axis=1)  # Shape: (N, 1, 64, 64)
y = np.array(labels, dtype=np.int64)

# Permute and split: 80% Train (480 scans), 20% Test (120 scans)
shuffled_indices = np.random.permutation(len(X))
split_idx = int(0.8 * len(X))
train_idx, test_idx = shuffled_indices[:split_idx], shuffled_indices[split_idx:]

train_X, train_y = torch.FloatTensor(X[train_idx]), torch.LongTensor(y[train_idx])
test_X, test_y = torch.FloatTensor(X[test_idx]), torch.LongTensor(y[test_idx])

train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=32, shuffle=False)

print(f"Dataset loaded: {len(X)} total images (Resolution: 64x64).")
print(f"Training set:   {len(train_X)} scans (Clean: {(train_y == 0).sum().item()}, Fouled: {(train_y == 1).sum().item()})")
print(f"Test set:       {len(test_X)} scans (Clean: {(test_y == 0).sum().item()}, Fouled: {(test_y == 1).sum().item()})")


In [ ]:
# Figure 1: Thermal Inspection Scans (Clean vs. Fouled Tubes)
fig, axes = plt.subplots(2, 4, figsize=(15, 6))

clean_indices = np.where(train_y.numpy() == 0)[0][:4]
fouled_indices = np.where(train_y.numpy() == 1)[0][:4]

# Top row: Clean tubes (Uniform axial temperature gradient)
for j, idx in enumerate(clean_indices):
    im = axes[0, j].imshow(train_X[idx, 0].numpy(), cmap='inferno', origin='lower')
    axes[0, j].set_title(f"Clean Tube #{j+1}", fontsize=11, fontweight='bold', color='darkgreen')
    axes[0, j].axis('off')
    fig.colorbar(im, ax=axes[0, j], fraction=0.046, pad=0.04)

# Bottom row: Fouled tubes (Localized thermal hotspot anomaly)
for j, idx in enumerate(fouled_indices):
    im = axes[1, j].imshow(train_X[idx, 0].numpy(), cmap='inferno', origin='lower')
    axes[1, j].set_title(f"Fouled Hotspot #{j+1}", fontsize=11, fontweight='bold', color='darkred')
    axes[1, j].axis('off')
    fig.colorbar(im, ax=axes[1, j], fraction=0.046, pad=0.04)

plt.suptitle('Infrared Thermography Scans: Clean Gradient vs. Fouling Hotspots', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## Part 2: CNN Architecture & Training

We use a compact 3-stage CNN:
1. **Block 1:** Conv2D (8 filters, $3\times 3$) $\to$ ReLU $\to$ MaxPool ($2\times 2$, $64 \to 32$).
2. **Block 2:** Conv2D (16 filters, $3\times 3$) $\to$ ReLU $\to$ MaxPool ($2\times 2$, $32 \to 16$).
3. **Block 3:** Conv2D (32 filters, $3\times 3$) $\to$ ReLU $\to$ MaxPool ($2\times 2$, $16 \to 8$).
4. **Classifier Head:** Linear ($32 \times 8 \times 8 = 2048 \to 32$) $\to$ ReLU $\to$ Linear ($32 \to 2$).

In [ ]:
# Define PyTorch CNN for thermal inspection diagnostics
class ThermalInspectionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional feature extraction backbone
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 64x64 -> 32x32
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        )
        # Decision classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2)  # Output classes: 0 = Clean, 1 = Fouled
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        logits = self.classifier(x)
        return logits

model = ThermalInspectionCNN()
print("Model Architecture Summary:\n", model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,} (lightweight, runs in seconds)")


<p align="center">
  <img src="./assets/cnn_architecture.png" alt="Thermal Inspection CNN Architecture" width="900">
</p>

In [ ]:
# Train the CNN model (executes in ~3-4 seconds on CPU)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

epochs = 15
train_losses = []
test_accuracies = []

print("Training CNN Classifier on Thermal Images...")
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
        
    epoch_loss = running_loss / len(train_X)
    
    # Evaluate accuracy on test scans
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            preds = model(batch_X).argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            
    test_acc = (correct / len(test_X)) * 100.0
    train_losses.append(epoch_loss)
    test_accuracies.append(test_acc)
    
    print(f"Epoch {epoch}/{epochs} | Training Loss: {epoch_loss:.4f} | Test Accuracy: {test_acc:.1f}%")

print("Training Complete!")


In [ ]:
# Figure 2: Visualizing what the CNN filters learn (Conv Block 1 feature maps)
fouled_sample = test_X[test_y == 1][0:1]  # Select one fouled test scan

model.eval()
with torch.no_grad():
    feature_maps = model.conv1(fouled_sample)  # Shape: (1, 8, 32, 32)

fig, axes = plt.subplots(1, 9, figsize=(18, 2.6))

# Display original raw input scan
axes[0].imshow(fouled_sample[0, 0].numpy(), cmap='inferno', origin='lower')
axes[0].set_title('Raw Input Scan', fontsize=10, fontweight='bold')
axes[0].axis('off')

# Display activations from each of the 8 convolutional kernels
for k in range(8):
    axes[k + 1].imshow(feature_maps[0, k].numpy(), cmap='magma', origin='lower')
    axes[k + 1].set_title(f'Filter #{k+1}', fontsize=10)
    axes[k + 1].axis('off')

plt.suptitle('Convolutional Kernel Activations: Isolating Hotspot Boundaries and Thermal Gradients', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# Figure 3: Automated Diagnostic Cards on Unseen Plant Test Scans
fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))

model.eval()
with torch.no_grad():
    sample_test_X = test_X[:6]
    logits = model(sample_test_X)
    probs = torch.softmax(logits, dim=1).numpy()
    preds = np.argmax(probs, axis=1)

for i in range(6):
    ax = axes[i]
    true_cls = test_y[i].item()
    pred_cls = preds[i]
    confidence = probs[i, pred_cls] * 100.0
    
    ax.imshow(sample_test_X[i, 0].numpy(), cmap='inferno', origin='lower')
    
    is_correct = (true_cls == pred_cls)
    status = "FOULED" if pred_cls == 1 else "CLEAN"
    color = "darkred" if pred_cls == 1 else "darkgreen"
    
    ax.set_title(f"Status: {status}\nConf: {confidence:.1f}%", fontsize=11, fontweight='bold', color=color)
    ax.axis('off')

plt.suptitle('Automated IR Diagnostic Cards on 6 Unseen Test Scans (Green = Clean, Red = Hotspot Detected)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()
